In [107]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [108]:
df = pd.read_csv('winequality-red.csv')
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [109]:
df.isnull().sum()  ##alredy clean data, no null values

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [110]:
X_train, X_test, y_train, y_test = train_test_split(
    df.iloc[:, :-1],   # all columns except last
    df.iloc[:, -1]-3,    # only last column ,i do -3 to make the quality start from 0 instead of 3
    test_size=0.2
)

In [111]:
X_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
1589,6.6,0.725,0.20,7.8,0.073,29.0,79.0,0.99770,3.29,0.54,9.2
877,7.7,0.715,0.01,2.1,0.064,31.0,43.0,0.99371,3.41,0.57,11.8
585,7.6,0.510,0.24,2.4,0.091,8.0,38.0,0.99800,3.47,0.66,9.6
1518,7.4,0.470,0.46,2.2,0.114,7.0,20.0,0.99647,3.32,0.63,10.5
1100,8.4,0.340,0.42,2.1,0.072,23.0,36.0,0.99392,3.11,0.78,12.4
...,...,...,...,...,...,...,...,...,...,...,...
1175,6.5,0.610,0.00,2.2,0.095,48.0,59.0,0.99541,3.61,0.70,11.5
1450,7.2,0.370,0.32,2.0,0.062,15.0,28.0,0.99470,3.23,0.73,11.3
1375,7.2,0.560,0.26,2.0,0.083,13.0,100.0,0.99586,3.26,0.52,9.9
345,7.0,0.685,0.00,1.9,0.067,40.0,63.0,0.99790,3.60,0.81,9.9


In [112]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1279, 11), (320, 11), (1279,), (320,))

In [113]:
y_train.unique()

array([2, 3, 5, 4, 0, 1])

In [114]:
# scaling the data
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [115]:
X_train.shape

(1279, 11)

In [116]:
# to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

In [117]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1279, 11), (320, 11), (1279,), (320,))

In [118]:
## datasets & dataloaders
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = features
    self.labels = labels

  def __len__(self):

    return self.features.shape[0]

  def __getitem__(self, index):

    return self.features[index], self.labels[index]


In [119]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [120]:
train_loader = DataLoader(train_dataset, batch_size=35, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=35, shuffle=False)

In [121]:
## defining the model
class MySimpleNN(nn.Module):

  def __init__(self, num_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(in_features=num_features, out_features=128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 16),
        nn.ReLU(),
        nn.Linear(16, 6),     # 6 neurons → one per quality grade
        nn.Softmax(dim=1)    # converts to 6 probabilities that sum to 1
    )

  def forward(self, features):
    y_pred = self.network(features)
    return y_pred



In [122]:
# set learning rate and epochs
epochs = 100
learning_rate = 0.1

In [123]:
# loss
loss_fn = nn.CrossEntropyLoss()

In [124]:
### model initialization
model = MySimpleNN(num_features=X_train_tensor.shape[1])

#builtin optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [125]:
## tranning loop
for epoch in range(epochs):
    
    for batch_features, batch_labels in train_loader:
        
        # forward pass
        y_pred = model(batch_features)

        # loss calculation
        loss = loss_fn(y_pred.squeeze(), batch_labels)

        # zero gradients
        optimizer.zero_grad()

        # backward pass
        loss.backward()

        # update weights
        optimizer.step()

    # print loss after each epoch
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 1.7762352228164673
Epoch: 2, Loss: 1.7317575216293335
Epoch: 3, Loss: 1.584884762763977
Epoch: 4, Loss: 1.567323088645935
Epoch: 5, Loss: 1.4454419612884521
Epoch: 6, Loss: 1.4594234228134155
Epoch: 7, Loss: 1.486639380455017
Epoch: 8, Loss: 1.4490898847579956
Epoch: 9, Loss: 1.6119698286056519
Epoch: 10, Loss: 1.3366365432739258
Epoch: 11, Loss: 1.639952540397644
Epoch: 12, Loss: 1.5570752620697021
Epoch: 13, Loss: 1.5706238746643066
Epoch: 14, Loss: 1.2949267625808716
Epoch: 15, Loss: 1.4044345617294312
Epoch: 16, Loss: 1.4511833190917969
Epoch: 17, Loss: 1.5023653507232666
Epoch: 18, Loss: 1.3865970373153687
Epoch: 19, Loss: 1.5954962968826294
Epoch: 20, Loss: 1.6327357292175293
Epoch: 21, Loss: 1.4180066585540771
Epoch: 22, Loss: 1.5194106101989746
Epoch: 23, Loss: 1.4131860733032227
Epoch: 24, Loss: 1.3347347974777222
Epoch: 25, Loss: 1.3681120872497559
Epoch: 26, Loss: 1.3682935237884521
Epoch: 27, Loss: 1.5096153020858765
Epoch: 28, Loss: 1.373633861541748
Epoch:

In [126]:
model.eval()
accuracy_list = []
predictions_list = []

with torch.inference_mode():
    for batch_features, batch_labels in test_loader:
        y_pred = model(batch_features)

        predicted_labels = torch.argmax(y_pred, dim=1)  # shape: [35] not [35,6]

        predictions_list.extend(predicted_labels.tolist())  # no view(-1) needed

        accuracy = (predicted_labels == batch_labels).float().mean()
        accuracy_list.append(accuracy.item())

overall_predictions = np.array(predictions_list)
print(f'Predictions: {overall_predictions}')
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Overall Accuracy: {overall_accuracy * 100:.2f}%')

Predictions: [2 3 3 2 2 3 3 3 3 3 3 3 2 3 3 2 3 2 2 3 3 3 2 3 3 3 2 2 3 3 3 3 2 3 3 2 3
 3 2 2 2 2 2 2 3 2 2 3 2 3 3 3 3 3 3 3 3 3 2 3 3 2 2 3 2 3 2 3 2 3 3 3 3 2
 2 3 2 2 2 2 2 2 2 3 2 2 3 2 3 2 3 3 3 3 3 3 2 2 3 3 3 2 3 3 2 3 2 2 3 2 3
 3 3 2 3 3 3 2 3 3 3 2 2 3 2 3 3 2 3 3 2 3 2 3 3 3 2 3 2 3 2 2 3 2 3 3 3 2
 3 2 3 2 2 3 3 2 3 3 2 3 3 3 2 2 3 3 2 3 3 3 3 2 3 2 3 2 3 3 3 3 3 2 3 3 2
 2 3 2 3 2 3 2 3 2 3 3 2 3 3 3 3 3 3 2 3 3 3 3 3 2 2 2 3 3 3 3 3 2 3 3 3 3
 2 2 2 2 3 3 3 3 3 3 3 2 2 3 2 2 2 3 2 3 2 3 3 3 3 3 2 2 3 2 3 3 2 3 2 3 3
 2 3 3 2 3 2 2 3 3 3 2 2 3 2 3 2 3 3 2 3 3 2 2 3 2 3 2 2 3 3 2 3 3 2 3 2 3
 2 2 2 3 2 2 2 3 2 3 3 3 2 3 2 2 2 3 3 3 3 3 2 2]
Overall Accuracy: 60.29%


In [127]:
# ───────────────────────────────────────────────
# Predict Wine Quality for new input
# ───────────────────────────────────────────────

# Column order:
# fixed acidity, volatile acidity, citric acid, residual sugar,
# chlorides, free sulfur dioxide, total sulfur dioxide,
# density, pH, sulphates, alcohol

new_wine = np.array([[
    7.4,     # fixed acidity
    0.70,    # volatile acidity
    0.00,    # citric acid
    1.9,     # residual sugar
    0.076,   # chlorides
    11.0,    # free sulfur dioxide
    34.0,    # total sulfur dioxide
    0.9978,  # density
    3.51,    # pH
    0.56,    # sulphates
    9.4      # alcohol
]], dtype=np.float32)

# scale ALL columns (unlike loan project, here everything is scaled)
new_wine_scaled = scaler.transform(new_wine)

# convert to tensor
input_tensor = torch.from_numpy(new_wine_scaled).float()

c:\Users\gamin\OneDrive\Desktop\DL_Projects\env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [128]:
# predict
model.eval()
with torch.inference_mode():
    output = model(input_tensor)               # shape: [1, 6]
    predicted_class = torch.argmax(output, dim=1).item()  # 0-5
    actual_quality = predicted_class + 3       # shift back → 3-8

print(f'Raw class (0-5)   : {predicted_class}')
print(f'Wine Quality Grade: {actual_quality}')
print(f'Rating            : {"⭐" * actual_quality}')

Raw class (0-5)   : 2
Wine Quality Grade: 5
Rating            : ⭐⭐⭐⭐⭐
